# Simulations in crypt-like geometries

Packages used:

In [ ]:
using Pkg
using CellBasedModels 
using GeometryBasics
using Distributions
using GLMakie, Colors
using CSV, DataFrames, Statistics
using Printf, JLD2
using SpecialFunctions
using LsqFit
using LinearAlgebra
using StatsBase
using Images, FileIO
using Graphs

First, a number of matrices are generated to control bacterial and medium movement with a crypt-like geometry. The depth, and width of these invaginations can be controlled with the correspondant parameters. 

In [ ]:
# =======================
# Function for crypt-like invaginations
# =======================

function wall_y(x, w, d, y0, xstart, xend)
    
    if x < xstart || x > xend
        return y0
    end

    ξ = x - xstart

    return y0 - d * floor((1 + sign(sin(2π * ξ / w))) / 2)
end

# ========================
# Generation of matrices
# ========================

x1, x2, y1, y2 = 0, 500, 0, 500
med1, med2 = 250, 250

width = 62
depth = 200
y_start = y2/2
x_start = 20
x_end = x2
dx = (x2 - x1) / med1
dy = (y2 -y1) / med2

xcoord(i1_) = x1 + (i1_ - 1) * dx
ycoord(i2_) = y1 + (i2_ - 1) * dy
Nx = med1
Ny = med2

mask_raw = zeros(Bool, med1, med2)
M0 = zeros(Bool, med1, med2)
MXL = zeros(Bool, med1, med2)
MXR = zeros(Bool, med1, med2)
MYT = zeros(Bool, med1, med2)

for i in 1:Nx, j in 1:Ny
    x = xcoord(i)
    y = ycoord(j)
    mask_raw[i,j] = y < wall_y(x, width, depth, y_start, x_start, x_end)
end

for i in 2:Nx-1, j in 2:Ny-1
    if mask_raw[i,j] &&
    mask_raw[i+1,j] &&
    mask_raw[i-1,j] &&
    mask_raw[i,j+1] &&
    mask_raw[i,j-1] &&
    mask_raw[i+1,j+1] &&
    mask_raw[i+1,j-1] &&
    mask_raw[i-1,j+1] &&
    mask_raw[i-1,j-1]

        M0[i,j] = true
    end
end

M0[1,   :] .= true
M0[250, :] .= true
M0[1,   125:end] .= false
M0[250, 125:end] .= false
M0[:, 1] .= true

for i in 2:Nx-1, j in 2:Ny-1

    if M0[i,j] == 1

                # --- HORIZONTAL LINES ---
                # bottom edge (neighbor below is outside)
        if M0[i, j-1] == 0
            MYT[i,j] = true
        end

                # top edge (neighbor above is outside)
        if M0[i, j+1] == 0
            MYT[i,j] = true
        end


                # --- LEFT WALL ---
        if M0[i-1, j] == 0
            MXL[i,j] = true
        end


                # --- RIGHT WALL ---
        if M0[i+1, j] == 0
            MXR[i,j] = true
        end

    end
end

wall_y (generic function with 1 method)

Next, the model is defined with the correspondant boundry conditions. In this case, a crypt attractant is added, as well as a self-attractant. 

In [ ]:
crypt_o2 = ABM(2,
    agent = Dict(
        :vx => Float64,
        :vy => Float64,
        :v => Float64, 
        :theta => Float64,
        :d => Float64,
        :l => Float64,
        :m => Float64,
        :active => Bool,

        :S => Float64,
        :S_o => Float64,

        :methyl => Float64,
        :Yp => Float64, 
        :G => Float64,
        :λ => Float64,
        :P => Float64,
        :M => Float64,
        :F => Float64,
        :A => Float64
    ),

    model = Dict(

        :Dr_run => Float64,

        :ε0 => Float64,
        :ε1 => Float64,
        :ε2 => Float64,
        :ε3 => Float64,
        :K => Float64,
        :Nrec => Float64, 
        :Ki => Float64,
        :Ka => Float64,
        :τm => Float64,
        :α => Float64,    
        :ωFrec => Float64,     
        :Ky => Float64,        
        :Z => Float64,          
        :Kz => Float64,       
        :Yy => Float64,  
        :DMedium => Float64,
        :delta => Float64,
        :DMedium_o => Float64,
        :delta_o => Float64
    ),

    medium = Dict(
        :mm => Float64,
        :mm_o => Float64,
        :ve => Float64
    ),

    agentODE = quote
  
        xmin, xmax = simBox[1,1], simBox[1,2]
        ymin, ymax = simBox[2,1], simBox[2,2]


        idx = Int(floor(Int, x/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)
        idy = Int(floor(Int, y/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)

        if x < xmin
            idx = Int(floor(Int, (x+(xmax - xmin))/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)

        elseif x > xmax
            idx = Int(floor(Int, (x-(xmax - xmin))/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)

        end

        if y < ymin
            idy = Int(floor(Int,(y+ymax-ymin)/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)

        elseif y > ymax
            idy = Int(floor(Int,(y-(ymax-ymin))/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)

        end

        mmb = max(0, mm[idx,idy])
        mmc = max(0, mm_o[idx,idy])

        F = ε0 + ε1 * methyl + Nrec * log((1 + mmb / Ki) / (1 + mmb / Ka)) + Nrec * log((1 + mmc / Ki) / (1 + mmc / Ka))
        F0 = log(((Ky * (α - K)) / (K * (Kz * Z + Yy))) - 1)      

        mx = (ε0 + Nrec * log((1 + mmb / Ki) / (1 + mmb / Ka)) + Nrec * log((1 + mmc / Ki) / (1 + mmc / Ka))- F0) / (- ε1)
       

        A = 1 / (1 + exp(F))    

        Yp = (Ky * A * α) / ((Ky * A) + (Kz * Z) + Yy)

        G = ε2 / 4 - (ε3 / 2) / (1 + (K / Yp))     

        dt(x) = vx 
        dt(y) = vy  
        dt(methyl) = -(1 / τm) * (methyl - mx)     
        
    end,

    agentRule = quote

        xmin, xmax = simBox[1,1], simBox[1,2]
        ymin, ymax = simBox[2,1], simBox[2,2]

        idx = Int(floor(Int, x/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)
        idy = Int(floor(Int, y/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)        
        
        # ==================
        # Agent influx 
        # ==================
        p_add = 0.005
        if i1_ == 1
            for k in 1:10
                if rand() < p_add 
                    @addAgent(      
                    x = xmin + 2,      
                    y = ymax/2 + 10 + rand()*(ymax - (ymax/2) - 10),       
                    theta = 2 * pi,     
                    l = 3    
                    )
                end      
            end
        end

        # ==================
        # Tumbling logic
        # ==================

        v_run = v
        v_tumble = 0.25 
        speed = active ? v_run : v_tumble

        Dr_tumble = 6.2      
        Dr_total = active ? Dr_run : Dr_tumble

        mm[idx,idy] += S

        if active 
            λ = ωFrec*exp(-G)
            P = 1 - exp(-λ * dt)
                
        else
            λ = ωFrec*exp(G)
            P = 1 - exp(-λ * dt)  
                
        end


        if active 
            λrt = ωFrec*exp(-G) 
            P_rt = 1 - exp(-λrt * dt)
            P = rand() 
                                                  
            if P < P_rt             
                active = false
                vx = speed* cos(theta) + ve[idx, idy]
                vy = speed* sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn() 
                        
            else    
                active = true
                vx = speed * cos(theta) + ve[idx, idy]
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()
                     
            end

        else
            λtr = ωFrec*exp(G) 
            P_tr = 1 - exp(-λtr * dt)
            P = rand()

            if P < P_tr
                active = true
                vx = speed * cos(theta) + ve[idx, idy]
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()

            else
                active = false
                vx = speed* cos(theta) + ve[idx, idy]
                vy = speed* sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()

            end
        end

        # ===================
        # Agent boundry conditions
        # ===================´

        if x < xmin
            x = xmin + (xmin - x)
            theta = pi - theta
        elseif x > xmax
            @removeAgent()
        end

        if y >= ymax
            y = 2*ymax - y
            theta = 2*pi - theta
        end

        if y <= wall_y(x, 62, 200, 250, 20, 500)
            x_new = clamp(round(Int, x/2), 1, 250)
            y_new = clamp(round(Int, y/2), 1, 250)

            if MYT[x_new, y_new] == 1 && MXR[x_new, y_new] == 1
                y = wall_y(x, 62, 200, 250, 20, 500) + 1
                x = x + 1
                theta = theta + pi

            elseif MYT[x_new, y_new] == 1 && MXL[x_new, y_new] == 1
                y = wall_y(x, 62, 200, 250, 20, 500) + 1
                x = x - 1
                theta = theta + pi

            elseif MYT[x_new, y_new] == 1 
                y = wall_y(x, 62, 200, 250, 20, 500) + 1
                theta = 2*pi - theta

            elseif MXR[x_new, y_new] == 1
                x = x + 2
                theta = pi - theta

            elseif MXL[x_new, y_new] == 1
                x = x - 2
                theta = pi - theta

            elseif M0[x_new, y_new] == 1        # Esta dins, no pinta res allí
                t0 = round(Int, (t-1))
                x_old = x[t0]
                y_old = y[t0]
                nsteps = ceil(Int, max(abs(x - x_old), abs(y - y_old)))

                hit_type = nothing

                for s in 0:nsteps
                    xs = x_old + (x - x_old) * s / nsteps
                    ys = y_old + (y - y_old) * s / nsteps

                    xi = clamp(round(Int, xs/2), 1, 250)
                    yi = clamp(round(Int, ys/2), 1, 250)

                    if MXR[xi, yi]
                        hit_type = :right
                        break
                    elseif MXL[xi, yi]
                        hit_type = :left
                        break
                    elseif MYT[xi, yi]
                        hit_type = :horizontal
                        break
                    end
                end
                if hit_type == :right
                    x = x_old
                    theta = pi - theta

                elseif hit_type == :left
                    x = x_old
                    theta = pi - theta

                elseif hit_type == :horizontal
                    y = y_old
                    theta = 2*pi - theta
                end
                                
            end
        end
    end,


    mediumODE = quote

        if @mediumInside()
            if M0[i1_, i2_]
                dt(mm) = 0
                dt(mm_o) = 0

            else
                dt(mm) = DMedium * (@∂2(1, mm) + @∂2(2, mm)) - delta * mm - ve[i1_, i2_] * @∂(1, mm)
                dt(mm_o) = DMedium_o * (@∂2(1, mm_o) + @∂2(2, mm_o)) - delta_o * mm_o - ve[i1_, i2_] * @∂(1, mm_o)

            end
            
            mm = MXR[i1_, i2_] ? mm[i1_ + 1, i2_] : mm
            mm = MXL[i1_, i2_] ? mm[i1_ - 1, i2_] : mm
            mm = MYT[i1_, i2_] ? mm[i1_, i2_ + 1] : mm

            # ======================
            # Crypt attractant generation
            # ======================
            mm_o = MXR[i1_, i2_] ? S_o : mm_o
            mm_o = MXL[i1_, i2_] ? S_o : mm_o
            mm_o = MYT[i1_, i2_] ? S_o : mm_o
            
        elseif @mediumBorder(1,-1)
            mm = 0
            mm_o = 0
        elseif @mediumBorder(1,1) 
            mm = 0
            mm_o = 0

        elseif @mediumBorder(2,-1) 
            mm = 0
            mm_o = 0

        elseif @mediumBorder(2,+1)
            mm = mm[i1_, NMedium[2] - 2]  
            mm_o = mm_o[i1_, NMedium[2] - 2]  

        end
    end,


    agentAlg = CBMIntegrators.Heun(),
    mediumAlg=DifferentialEquations.Heun()
)

Then, for different self-attractant dimensionless run length parameters, the community is loaded and the simulations evolved for 50000 steps. 

In [ ]:

steps = 50000
ns = [0.5, 3.0, 8.0]


for (idx, n) in enumerate(ns)
    println("Running  n = 0.25")

    com = Community(
        crypt_o2,
        N=10,
        dt=0.01,
        simBox = [0.0 500.0; 0.0 500.0],
        NMedium = [250,250]
    )

    m = 1/100
    g = 1/10000
    d = 1

    com.Dr_run = 0.062

    com.v = 10.0

    com.ωFrec = 1.3
    com.Ki = 0.0182
    com.Ka = 3.0
    com.Nrec = 6.0
    com.ε0   = 6.0
    com.ε1   = -1.0
    com.ε2   = 80
    com.ε3   = 80

    com.τm = 1

    com.α   = 6.0
    com.K = 2.0 
    com.Ky = 100.0
    com.Kz = 10.0
    com.Z = 5.0
    com.Yy = 0.1

    com.m = 1.        
    com.d = 1.        
    com.l = 3;

    Dc = 10
    delta = 0.01

    com.DMedium_o = 20
    com.DMedium = Dc / n
    com.delta_o = 0.005
    com.delta = delta * n


    com.x = 5
    com.y = 400

    com.theta = rand(Uniform(0,2*pi),com.N)

    com.methyl .= 0.0
    com.Yp .= com.K

    com.active .= true 

    com.S = 0.0025
    com.S_o = 1

    # =============
    # Difference of advection between lumen and near crypts regions
    # =============

    ve = 5

    com.ve = zeros(com.NMedium[1], com.NMedium[2])

    L = 50   

    for i1 in 1:com.NMedium[1], i2 in 1:com.NMedium[2]
        y = ycoord(i2)

        if y <= 250
            com.ve[i1, i2] = 0.0

        elseif y < 250 + L
            f = (y - 250) / L
            com.ve[i1, i2] = ve * f^2

        else
            com.ve[i1, i2] = ve
        end
    end

    # ==============
    # Initialization of crypt attractant - loaded from before
    # ==============
    com.mm_o = o2grid

    loadToPlatform!(com, preallocateAgents=10000)
        
    outfile = @sprintf("c_o2_selfpos_%s.jld2", n)  ###CANVIARRR NOOM

    jldopen(outfile, "w") do file

        for step in 1:steps
            CellBasedModels.step!(com)
            if step%100 == 0
                stepname = @sprintf("step_%06d", step)
                g = JLD2.Group(file, stepname)

                g["x"] = copy(com.x)
                g["y"] = copy(com.y)
                g["theta"] = copy(com.theta)

                g["mm_grid"] = copy(com.mm)
                if step%5000 == 0
                    println(step)
                    
                end
            end
        end
    end
end

## Analysis

The following code was used to extract the number of agents in each crypt and the distribution along the crypts. 

In [ ]:
dfs = Dict{Float64, DataFrame}()


ns = [0.5, 3.0, 8.0]


for (n_idx, n) in enumerate(ns)
        
    outfile = "c_o2_selfpos_$N.jld2"
        

    data = Dict{Int, Any}()
    steps = 50000
    jldopen(outfile, "r") do file
        for step in 100:100:steps
            key = file[@sprintf("step_%06d", step)]
            data[step] = Dict(
                "x" => copy(key["x"]),
                "y" => copy(key["y"])
            )
        end
    end

    x1, x2, y1, y2 = 0, 500, 0, 500
    med1, med2 = 250, 250

    width = 62
    depth = 200
    y_start = y2/2
    x_start = 20
    x_end = x2
    dx = (x2 - x1) / med1
    dy = (y2 -y1) / med2

    xcoord(i1_) = x1 + (i1_ - 1) * dx
    ycoord(i2_) = y1 + (i2_ - 1) * dy
    Nx = med1
    Ny = med2

    mask_raw = zeros(Bool, med1, med2)
    M0 = zeros(Bool, med1, med2)
    MXL = zeros(Bool, med1, med2)
    MXR = zeros(Bool, med1, med2)
    MYT = zeros(Bool, med1, med2)

    for i in 1:Nx, j in 1:Ny
        x = xcoord(i)
        y = ycoord(j)
        mask_raw[i,j] = y < wall_y(x, width, depth, y_start, x_start, x_end)
    end

    for i in 2:Nx-1, j in 2:Ny-1
        if mask_raw[i,j] &&
        mask_raw[i+1,j] &&
        mask_raw[i-1,j] &&
        mask_raw[i,j+1] &&
        mask_raw[i,j-1] &&
        mask_raw[i+1,j+1] &&
        mask_raw[i+1,j-1] &&
        mask_raw[i-1,j+1] &&
        mask_raw[i-1,j-1]

            M0[i,j] = true
        end
    end

    for i in 2:Nx-1, j in 2:Ny-1

        if M0[i,j] == 1

            # --- HORIZONTAL LINES ---
            # bottom edge (neighbor below is outside)
            if M0[i, j-1] == 0
                MYT[i,j] = true
            end

            # top edge (neighbor above is outside)
            if M0[i, j+1] == 0
                MYT[i,j] = true
            end


            # --- LEFT WALL ---
            if M0[i-1, j] == 0
                MXL[i,j] = true
            end


                # --- RIGHT WALL ---
            if M0[i+1, j] == 0
                MXR[i,j] = true
            end
        end
    end


    ncripts = 8
    steps_t = 100:100:steps
    sizes = length(steps_t)
    counts = zeros(ncripts, sizes)
    agents = Float64[]
    counts_in = Float64[]
    counts_out = Float64[]

    density_lumen = Float64[]
    density_crypt = Float64[]

    residence_time = Float64[]

    occupancy = Float64[]

    for i in 1:sizes
        count_in = 0
        count_out = 0
        agent = 0
        step = steps_t[i]
        g = data[step]
        x = g["x"]
        y = g["y"]
            
        for j in 1:length(x)

            if y[j] < 250 && y[j] != 0
                count_in += 1
                agent += 1

            elseif y[j] > 250 && y[j] != 0
                count_out += 1
                agent += 1

            end          
        end

        area_lumen = 250 * 500
        area_crypt = ncripts * (62/2) * 200 ## ncripts * width/2 * depth
        d_l = count_out / area_lumen
        d_c = count_in / area_crypt
        percentatge_in = (count_in / agent) * 100

        push!(counts_in, count_in)
        push!(counts_out, count_out)
        push!(agents, agent)
        push!(density_lumen, d_l)
        push!(density_crypt, d_c)
        push!(occupancy, percentatge_in)
    end
        
    right_wall = findall(MXR[:, 100] .== 1) .* 2
    left_wall  = findall(MXL[:, 100] .== 1) .* 2

    n_bins = 8
    y_top = y_start
    y_bottom = y_start - depth

    n_steps = length(steps_t)
    counts = zeros(n_bins, n_steps)

    x_left  = right_wall[1:n_bins]
    x_right = left_wall[2:n_bins+1]
        
    for (i, step) in enumerate(steps_t)
        g = data[step]
        x = g["x"]
        y = g["y"]

        local_counts = zeros(Int, n_bins)

        for j in eachindex(x)
            if y[j] >= y_top
                continue
            end

            xj = x[j]

            for b in 1:n_bins
                if x_right[b] > xj > x_left[b]
                    local_counts[b] += 1
                    break
                end
            end
        end

        counts[:, i] .= local_counts
    end

    df = DataFrame(
        time_vec = sizes,
        num_agents = agents,
        occupancies = occupancy,
        d_lumen = density_lumen,
        d_crypt = density_crypt,
        num_inside = counts_in,
        num_outside = counts_out, 
        crypt_1 = counts[1, :],
        crypt_2 = counts[2, :],
        crypt_3 = counts[3, :],
        crypt_4 = counts[4, :],
        crypt_5 = counts[5, :],
        crypt_6 = counts[6, :],
        crypt_7 = counts[7, :],
        crypt_8 = counts[8, :]
    )

        
    dfs[n] = df
end



In [ ]:
function shannon_entropy(counts)

    total = sum(counts)

    total == 0 && return 0.0

    p = counts ./ total

    H = 0.0

    for pi in p
        if pi > 0
            H -= pi * log(pi)
        end
    end

    return H
end


An example usage once different simulations are done to compare the distribution along the crypts by calculating the normalized shannon entropy at the last time step.

In [ ]:
ns = [0.5, 3.0, 8.0]

o2_runs  = [dfs_o2_1, dfs_o2_2, dfs_o2_3, dfs_o2_4]
add_runs = [dfs_add_1, dfs_add_2, dfs_add_3, dfs_add_4]
o2_neg = [dfs_o2_neg, dfs_o2_neg_2, dfs_o2_neg_3]
non_chemo = [dfs_non_chemo[1], dfs_non_chemo[2], dfs_non_chemo[3]]
o2_only = [dfs_o2_only[1], dfs_o2_only[2], dfs_o2_only[3]]

step_idx = 500 

o2_vals  = [Float64[] for _ in ns]
add_vals = [Float64[] for _ in ns]
o2_neg_vals = [Float64[] for _ in ns]
non_vals = Float64[]
o2_only_vals = Float64[]

for run in o2_only
    df = run
    crypt_matrix = Matrix(df[:, [:crypt_1,:crypt_2,:crypt_3,:crypt_4,
                             :crypt_5,:crypt_6,:crypt_7,:crypt_8]])
    vector = crypt_matrix[500, :]
    
    entropy = shannon_entropy(vector)
    
    norm_H = entropy ./ log(8)
   
    push!(o2_only_vals, norm_H)    
end       

for run in non_chemo
    df = run
    crypt_matrix = Matrix(df[:, [:crypt_1,:crypt_2,:crypt_3,:crypt_4,
                             :crypt_5,:crypt_6,:crypt_7,:crypt_8]])
    vector = crypt_matrix[500, :]
    
    entropy = shannon_entropy(vector)
    
    norm_H = entropy ./ log(8)
   
    push!(non_vals, norm_H)    
end   
                             

for (i, n) in enumerate(ns)

    # O2 replicates
    for run in o2_runs
        df = run[n]
        crypt_matrix = Matrix(df[:, [:crypt_1,:crypt_2,:crypt_3,:crypt_4,
                             :crypt_5,:crypt_6,:crypt_7,:crypt_8]])
        vector = crypt_matrix[500, :]
    
        entropy = shannon_entropy(vector)
    
        norm_H = entropy ./ log(8)
   
        push!(o2_vals[i], norm_H)
    end

    # Self attractant
    for run in add_runs
        df = run[n]
        crypt_matrix = Matrix(df[:, [:crypt_1,:crypt_2,:crypt_3,:crypt_4,
                             :crypt_5,:crypt_6,:crypt_7,:crypt_8]])
        vector = crypt_matrix[500, :]
    
        entropy = shannon_entropy(vector)
    
        norm_H = entropy ./ log(8)
   
        push!(add_vals[i], norm_H)
    end

    for run in o2_neg
        df = run[n]
        crypt_matrix = Matrix(df[:, [:crypt_1,:crypt_2,:crypt_3,:crypt_4,
                             :crypt_5,:crypt_6,:crypt_7,:crypt_8]])
        vector = crypt_matrix[500, :]
    
        entropy = shannon_entropy(vector)
    
        norm_H = entropy ./ log(8)
        push!(o2_neg_vals[i], norm_H)
    end
end